In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Data Cleaning Notebook Ready")

Data Cleaning Notebook Ready


In [2]:
# Path to raw datasets
raw_path = Path("../data/raw")

# Read every CSV file
datasets = {}

for file in raw_path.glob("*.csv"):
    name = file.stem
    df = pd.read_csv(file)

    # Convert DATE column
    df["DATE"] = pd.to_datetime(df["DATE"])

    datasets[name] = df

print(f"Loaded {len(datasets)} datasets successfully!")
print(list(datasets.keys()))

Loaded 10 datasets successfully!
['Consumer_Sentiment', 'CPI', 'Federal_Funds_Rate', 'Housing_Starts', 'Industrial_Production', 'Real_GDP', 'Recession', 'Treasury_10Y', 'Treasury_2Y', 'Unemployment']


In [3]:
# Missing values report
missing_report = pd.DataFrame()

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_report[name] = missing

missing_report.T

,DATE,Consumer_Sentiment
Consumer_Sentiment,0.0,210.0
CPI,0.0,NaN
Federal_Funds_Rate,0.0,NaN
Housing_Starts,0.0,NaN
Industrial_Production,0.0,NaN
Real_GDP,0.0,NaN
Recession,0.0,NaN
Treasury_10Y,0.0,NaN
Treasury_2Y,0.0,NaN
Unemployment,0.0,NaN


In [4]:
from functools import reduce

# Merge all datasets on DATE
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on="DATE", how="outer"),
    datasets.values()
)

# Sort by date
merged_df = merged_df.sort_values("DATE").reset_index(drop=True)

print("Merged Shape:", merged_df.shape)
merged_df.head()

Merged Shape: (2061, 11)


,DATE,Consumer_Sentiment,CPI,Federal_Funds_Rate,Housing_Starts,Industrial_Production,Real_GDP,Recession,Treasury_10Y,Treasury_2Y,Unemployment
0,1854-12-01,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
1,1855-01-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,1855-02-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,1855-03-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,1855-04-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [5]:
# Check missing values after merging
merged_df.isnull().sum().sort_values(ascending=False)

Real_GDP                 1743
Treasury_2Y              1458
Consumer_Sentiment       1386
Housing_Starts           1249
Federal_Funds_Rate       1195
Treasury_10Y             1180
Unemployment             1118
CPI                      1106
Industrial_Production     769
DATE                        0
Recession                   0
dtype: int64

In [6]:
# Fill missing values
merged_df = merged_df.ffill().bfill()

# Check again
merged_df.isnull().sum()

DATE                     0
Consumer_Sentiment       0
CPI                      0
Federal_Funds_Rate       0
Housing_Starts           0
Industrial_Production    0
Real_GDP                 0
Recession                0
Treasury_10Y             0
Treasury_2Y              0
Unemployment             0
dtype: int64

In [7]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

merged_df.to_csv(processed_path / "economic_data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
